# 02 · Análisis exploratorio (EDA) de Work-Olist

Describir variables, identificar asimetría y variabilidad, reconocer faltantes y anomalías, explorar asociaciones sin confundir correlación y causalidad.

**Modalidad:** práctica guiada, código ejecutable, interpretación y ejercicios. **Datos:** nueve CSV aportados por el usuario en `data/raw/`, con el esquema relacional de Olist. Se calculan resultados sobre estos archivos, sin simular registros ni cambiar las fuentes.

## Objetivos
Caracterizar la unidad de análisis; revisar metadatos y calidad; construir tablas descriptivas; comparar media y mediana; estudiar cuartiles y asimetría; explorar relaciones bivariadas; generar preguntas, no conclusiones causales. Los conceptos vienen de las notas históricas sobre naturaleza, escalas, distribución, EDA, correlación y valores extremos.

**Datos y advertencia:** este cuaderno detecta automáticamente los CSV reales completos en `data/raw/`. Si faltan archivos, detiene la ejecución con un error descriptivo. Las conclusiones serán condicionales al archivos entregados. Los precios están en reales brasileños (BRL) y el precio de ítems excluye flete.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import numpy as np
import pandas as pd
# Detectar carpeta del proyecto desde raíz/ o notebooks/ (sin rutas absolutas).
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "olist_tools.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecute el notebook dentro de la carpeta Work_Olist_Notebooks_2026 (README.md).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.olist_tools import (read_tables, build_order_level, build_item_level,
                            table_inventory, missing_profile, safe_minmax, safe_zscore)
TABLAS, ORIGEN = read_tables(ROOT)
print("ORIGEN:", ORIGEN)
print("Directorio de trabajo:", ROOT)

In [ ]:
orders = build_order_level(TABLAS)
items = build_item_level(TABLAS)
print(f"Unidad de la tabla principal: pedido; {len(orders):,} pedidos.")
display(table_inventory(TABLAS))

### Primera lectura verificada de la base

En los CSV aportados hay **99 441 pedidos**, **96 096 identificadores de compradores distintos**, **112 650 líneas de artículos**, y **99 224 registros de reseñas**. La cobertura de compras se extiende desde septiembre de 2016 hasta octubre de 2018; los meses de los extremos pueden estar incompletos. No se conoce aquí el diseño de muestreo de la plataforma ni la representatividad nacional.

En `orders`, los NA de `order_delivered_customer_date` suman **2 965**; no deben imputarse automáticamente porque el pedido podría no haberse entregado. Las categorías de producto tienen **610 valores ausentes** en `products`. Estas cifras se obtienen de los archivos recibidos y también pueden recalcularse con las celdas siguientes.


## 1. ¿Qué población describe esta muestra?
Se dispone de registros de una plataforma comercial. La unidad es el pedido; los datos no forman necesariamente una muestra probabilística de todo el comercio electrónico de Brasil. `customer_id` no es equivalente a un individuo único. Los códigos de estado y tipo de pago son **cualitativos nominales**; la puntuación 1–5 es **ordinal** y, aunque se promedie por conveniencia, requiere cautela.

In [ ]:
variables = pd.DataFrame([
 ["order_id","Identificador","nominal","Pedido"],
 ["customer_unique_id","Identificador","nominal","Comprador anonimizado"],
 ["subtotal_productos","BRL","cuantitativa de razón","Suma de precios de ítems"],
 ["dias_hasta_entrega","días","cuantitativa de razón","Solo entregados con fechas válidas"],
 ["puntuacion_resena","1–5","ordinal","Reseña observada; puede faltar"],
 ["order_status","categoría","nominal","Estado del pedido"],
 ["customer_state","código","nominal","Estado geográfico"],
 ], columns=["variable","unidad","escala","observación"])
display(variables)

## 2. Completitud y calidad preliminar
Primero observar el contexto de los NA. Una fecha de entrega faltante para un pedido cancelado es distinta de un precio ausente por error de captura. La presencia de datos faltantes debe cuantificarse antes de elegir tratamiento; la limpieza se realizará en el cuarto notebook.

In [ ]:
perfil=missing_profile(orders)
display(perfil.head(18))
print("Pedidos duplicados por llave:",orders["order_id"].duplicated().sum())
print("Fechas de entrega negativas (revisar, no borrar automáticamente):",
      int(orders["dias_hasta_entrega"].lt(0).sum()))
print("Ítems con precio <= 0:", int(TABLAS["items"]["price"].le(0).sum()))

## 3. Frecuencias y tablas categóricas
Para una variable nominal describimos frecuencias y proporciones. No se deben promediar códigos postales ni ordenar categorías por supuesta magnitud. Considera tanto conteos como denominadores.

In [ ]:
def frecuencias(s):
    n=s.value_counts(dropna=False).rename('conteo')
    return pd.concat([n, (n/n.sum()).rename('proporcion')],axis=1)
display(frecuencias(orders['order_status']).round(3))
display(frecuencias(orders['medio_pago_inicial']).round(3))
display(frecuencias(orders['puntuacion_resena']).round(3))

## 4. Centro, dispersión y posición
Media muestral $\bar{x}=\frac{1}{n}\sum_i x_i$; varianza muestral $s^2=\frac{1}{n-1}\sum_i(x_i-\bar{x})^2$. Mediana y rango intercuartílico (RIC = Q3 − Q1) resisten mejor la influencia de unos pocos valores extremos. `describe` usa convenciones de cuantiles que debemos documentar.

In [ ]:
medidas = ['subtotal_productos','total_flete','n_items','dias_hasta_entrega','dias_retraso']
display(orders[medidas].describe(percentiles=[.05,.25,.5,.75,.95,.99]).round(2))
resumen=[]
for nombre in medidas:
    s=orders[nombre].dropna()
    resumen.append({'variable':nombre,'n':s.size,'media':s.mean(),'mediana':s.median(),
                    'desv_std_muestral':s.std(ddof=1),'RIC':s.quantile(.75)-s.quantile(.25),
                    'asimetria':s.skew()})
display(pd.DataFrame(resumen).round(2))

**Pregunta interpretativa.** ¿Por qué el valor medio de un pedido puede diferir considerablemente del típico? Una distribución de precios comerciales suele ser asimétrica; no debemos presuponerlo sin mirar los cuantiles y una gráfica. Si aparece una observación extrema, puede ser real o un error: el diagnóstico no equivale a eliminación.

In [ ]:
precio=orders['subtotal_productos'].dropna()
if not precio.empty:
    print(f"Media={precio.mean():,.2f} BRL; mediana={precio.median():,.2f} BRL; "
          f"Q95={precio.quantile(.95):,.2f} BRL; máximo={precio.max():,.2f} BRL")
    print("Interpretación: la distancia media-mediana aporta pistas sobre asimetría, no demuestra causalidad.")

## 5. Distribución condicional y estratificación
El promedio global puede ocultar comportamientos por región, categoría, periodo o estado del pedido. La tabla siguiente separa los pedidos **entregados** y calcula medidas por estado. Cuidado: distintos tamaños y cobertura pueden afectar las comparaciones.

In [ ]:
entregados = orders.loc[orders['order_status'].eq('delivered')].copy()
por_region=(entregados.groupby('customer_state',dropna=False)
    .agg(n_pedidos=('order_id','nunique'),mediana_precio=('subtotal_productos','median'),
         media_entrega_dias=('dias_hasta_entrega','mean'),
         mediana_entrega_dias=('dias_hasta_entrega','median'),
         porcentaje_retrasos=('entrega_tardia','mean'))
    .sort_values('n_pedidos',ascending=False))
display(por_region.head(9).round(3))

## 6. Calendario de compras y denominadores
Los conteos mensuales dependen del intervalo cubierto por los datos. Un mes incompleto no debe compararse sin advertencia con uno completo. Los agregados de ingresos deben especificar la definición: aquí **precio de ítems** excluye flete y se basa en la tabla de pedidos agregada.

In [ ]:
mensual=(orders.dropna(subset=['mes_compra']).groupby('mes_compra')
         .agg(pedidos=('order_id','nunique'),valor_productos=('subtotal_productos','sum'),
              mediana_valor=('subtotal_productos','median')).sort_index())
display(mensual.head(12).round(2))

## 7. Asociación: Pearson y Spearman
Pearson cuantifica asociación **lineal**; Spearman usa rangos y puede reflejar asociaciones monótonas no necesariamente lineales. Ninguno prueba causalidad. Revisión adicional: dispersión, valores extremos, tamaño muestral, agrupación por segmentos y sesgos de selección.

In [ ]:
var_num = ['subtotal_productos','total_flete','n_items','dias_hasta_entrega','dias_retraso','puntuacion_resena']
display(orders[var_num].corr(method='pearson',numeric_only=True).round(2))
display(orders[var_num].corr(method='spearman',numeric_only=True).round(2))
casos=orders[['dias_retraso','puntuacion_resena']].dropna()
if len(casos)>2:
    print('n pares observados:',len(casos),
          'Pearson:',round(casos.corr(method='pearson').iloc[0,1],3),
          'Spearman:',round(casos.corr(method='spearman').iloc[0,1],3))

### Sensibilidad a un extremo: ejemplo matemático controlado
Usamos cinco puntos **inventados** para mostrar que una observación influyente puede cambiar el coeficiente de Pearson. No atribuimos este fenómeno a Olist sin diagnosticarlo en sus tablas reales.

**Este único ejemplo matemático es deliberadamente hipotético**, no un registro agregado a los CSV Olist. Se utiliza exclusivamente para mostrar por qué una correlación puede cambiar al introducir una observación influyente.

In [ ]:
ejemplo=pd.DataFrame({'x':[1,2,3,4,5,6], 'y':[1.3,2.0,3.1,4.0,5.2,5.9]})
con_extremo=pd.concat([ejemplo,pd.DataFrame({'x':[12],'y':[-9]})],ignore_index=True)
print('r sin extremo:',round(ejemplo.corr().loc['x','y'],3))
print('r con extremo:',round(con_extremo.corr().loc['x','y'],3))

## 8. Valores atípicos: criterio de IQR como señal, no sentencia
El intervalo $[Q_1-1.5 RIC,\ Q_3+1.5 RIC]$ proporciona una regla exploratoria. En precios muy asimétricos puede marcar muchos valores legítimos. **No eliminaremos** observaciones en este notebook; registraremos su proporción y su contexto.

In [ ]:
s=items['price'].dropna()
q1,q3=s.quantile([.25,.75]); ric=q3-q1
limite_inferior,limite_superior=q1-1.5*ric,q3+1.5*ric
bandera=(items['price']<limite_inferior)|(items['price']>limite_superior)
print(f'Límites exploratorios: [{limite_inferior:.2f}, {limite_superior:.2f}] BRL')
print('Marcados:',int(bandera.sum()),'de',int(items['price'].notna().sum()),'ítems con precio observado')
display(items.loc[bandera,['order_id','price','freight_value','categoria_mostrada']].head())

## 9. EDA de categorías: la unidad vuelve a importar
El catálogo se une a **ítems**, no directamente a pedidos, porque un pedido puede contener varias categorías. Por tanto, las cifras siguientes son por línea de producto y no por comprador. Si quieres el número de pedidos por categoría, cuenta `order_id` únicos después de filtrar la categoría.

In [ ]:
categorias=(items.groupby('categoria_mostrada',dropna=False)
    .agg(lineas=('order_id','size'),pedidos=('order_id','nunique'),
         valor_productos=('price','sum'),precio_mediano=('price','median'))
    .sort_values('lineas',ascending=False))
display(categorias.head(12).round(2))

### Interpretación guiada

Los resúmenes numéricos no sustituyen la comprensión de la unidad observacional. Un precio alto puede ser una venta genuina, un pedido puede no tener reseña y un pedido cancelado puede carecer de fecha de entrega por diseño. Para proponer una intervención, primero responde: **¿qué representa la columna, quién pudo registrar el dato y qué sesgo introduciría cambiarlo?**


## 10. Bitácora de hallazgos y límites
Escribe un párrafo por cada hallazgo con su indicador, su población (pedidos, ítems, compradores), su periodo y una limitación. Por ejemplo: «En los pedidos entregados y con fechas completas, la mediana de duración es ..., pero las filas sin fecha no se tratan como entrega instantánea».

**Ejercicios:**
- **Esencial:** mediana, RIC y porcentaje de NA para tres variables; muestra sus unidades.
- **Interpretación:** ¿por qué comparar las medias de dos estados sin controlar tamaño de muestra puede inducir errores?
- **Extensión:** contrasta correlación de Pearson y Spearman de `dias_retraso` con puntuación, con y sin un percentil extremo, **sin borrar los datos originales**.

**Referencias:** presentaciones históricas (análisis estadístico, EDA, valores faltantes); matriz docente; [pandas](https://pandas.pydata.org/docs/); [Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

In [ ]:
# Ejercicios: escriba aquí sus tres análisis. Analiza los resultados observados en los CSV suministrados.